# Scenario Lab: Build a Tiny Admission Predictor with NumPy

In this exercise notebook, you are a junior machine learning engineer helping a fictional university admissions team. They want a very small prototype model that predicts whether a student is likely to be admitted based on two features:
- exam score
- interview score

The team wants to understand the model mechanics, so you are **not allowed to use PyTorch or TensorFlow yet**. You must build the model step by step with **NumPy**.

Your final goal is to:
1. generate a simulated admissions dataset
2. build a single-layer perceptron baseline
3. study activation and loss functions
4. build a small MLP that performs better on a harder nonlinear version of the task

The notebook is organized as a sequence of tasks. Each task includes questions and coding work that help you reach the final goal.


## Scenario Overview

The admissions team gives you two modeling situations:

- **Phase A: Simple screening rule**
  Students are admitted if their exam and interview scores combine roughly linearly.
  A perceptron should work well here.

- **Phase B: Complex decision pattern**
  The team later decides that students with *balanced* scores should be preferred over students with only one extremely high score.
  This creates a nonlinear pattern that a single perceptron may struggle with.
  You will need an MLP for this phase.

As you work, answer the short prompts in the markdown cells and complete the `TODO` sections in the code cells.


## Task 0. Setup

Before starting, import the libraries and set the random seed.

Question:
- Why do we set a random seed in a teaching notebook?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')


## Task 1. Create the First Admissions Dataset

The first admissions officer uses a simple rule:

> Admit a student if a weighted combination of exam score and interview score is above a threshold.

This is a **linear decision rule**, so it is a good fit for a perceptron.

Questions:
- What kind of decision boundary do you expect from this rule?
- Why is this a good first problem for a perceptron?

Instructions:
- Complete the missing lines below to generate the linear scores and binary labels.


In [ ]:
n_students = 220
X_linear = np.random.randn(n_students, 2)
true_w = np.array([1.7, 1.1])
true_b = -0.2

# TODO 1: compute a linear score for each student
linear_scores = ...

# TODO 2: convert the scores into labels of shape (n_students, 1)
# label 1 means admitted, label 0 means not admitted
y_linear = ...

print('X_linear shape:', X_linear.shape)
print('y_linear shape:', y_linear.shape)
print('First five labels:', y_linear[:5].ravel())


## Task 2. Visualize the Admissions Data

Plot the students and color them by admission decision.

Questions:
- Does the dataset look linearly separable?
- If you had to draw one line to separate the classes, do you think it would work reasonably well?


In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(X_linear[:, 0], X_linear[:, 1], c=y_linear.ravel(), cmap='coolwarm', alpha=0.8)
plt.xlabel('Exam score (standardized)')
plt.ylabel('Interview score (standardized)')
plt.title('Admissions Dataset: Linear Rule')
plt.show()


## Task 3. Build the Perceptron Prediction Rule

A perceptron computes a score and then applies a threshold.

Mathematically:

$$z = XW + b$$

Then:
- predict 1 if $z \ge 0$
- predict 0 otherwise

Questions:
- What is the role of `W`?
- What is the role of `b`?
- Why is the step function enough for this simple classifier?


In [ ]:
def step_function(z):
    # TODO 3: return 1 where z >= 0, else 0
    return ...


def perceptron_forward(X, W, b):
    # TODO 4: compute the raw score z
    z = ...

    # TODO 5: convert scores into predictions
    y_hat = ...
    return z, y_hat


W_demo = np.random.randn(2, 1) * 0.1
b_demo = np.zeros((1,))
raw_scores, pred_labels = perceptron_forward(X_linear[:5], W_demo, b_demo)
print('Raw scores:\n', raw_scores)
print('Predicted labels:\n', pred_labels)


## Task 4. Train the Perceptron Baseline

Now you will train the admissions baseline model.

For one training example, the perceptron update is:

$$W \leftarrow W + \eta (y_i - \hat{y}_i)x_i$$

$$b \leftarrow b + \eta (y_i - \hat{y}_i)$$

Questions:
- What happens if the model prediction is already correct?
- What does the learning rate control?

Instructions:
- Fill in the missing lines for the error term and parameter updates.


In [ ]:
def train_perceptron(X, y, learning_rate=0.1, epochs=20):
    W = np.zeros((X.shape[1], 1))
    b = np.zeros((1,))
    history = []

    for epoch in range(epochs):
        mistakes = 0

        for i in range(X.shape[0]):
            xi = X[i:i+1]
            yi = y[i:i+1]
            _, y_hat = perceptron_forward(xi, W, b)

            # TODO 6: compute the prediction error
            error = ...

            if error.item() != 0:
                mistakes += 1

            # TODO 7: update the weights
            W += ...

            # TODO 8: update the bias
            b += ...

        _, preds = perceptron_forward(X, W, b)
        acc = np.mean(preds == y)
        history.append((mistakes, acc))

    return W, b, history


W_p, b_p, perceptron_history = train_perceptron(X_linear, y_linear, learning_rate=0.05, epochs=20)
print('Learned weights:', W_p.ravel())
print('Learned bias:', b_p)
print('Final training accuracy:', perceptron_history[-1][1])


## Task 5. Evaluate the Baseline Model

Plot how the number of mistakes and the training accuracy change over time.

Questions:
- Is the perceptron improving over epochs?
- Does the trend match what you expected from a linearly separable dataset?


In [ ]:
mistakes = [m for m, _ in perceptron_history]
accs = [a for _, a in perceptron_history]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(mistakes, marker='o')
axes[0].set_title('Perceptron Mistakes per Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Mistakes')

axes[1].plot(accs, marker='o', color='green')
axes[1].set_title('Perceptron Accuracy per Epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')

plt.tight_layout()
plt.show()


## Task 6. The Admissions Team Changes the Rule

The admissions team now says:

> We prefer students whose exam and interview scores are both reasonably strong and balanced.

This creates a more complex decision pattern. A single line may no longer be enough.

We will simulate this second scenario by labeling students near the center of a target region as admitted.

Questions:
- Why might a single-layer perceptron struggle here?
- What kind of model change could help?


In [ ]:
X_nonlinear = np.random.uniform(-2.0, 2.0, size=(320, 2))

# Distance from the origin acts like a simple proxy for balance of scores
radius = np.sqrt(np.sum(X_nonlinear**2, axis=1))

# TODO 9: label students near the center as admitted
# Hint: return labels with shape (320, 1)
y_nonlinear = ...

print('X_nonlinear shape:', X_nonlinear.shape)
print('y_nonlinear shape:', y_nonlinear.shape)


In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(X_nonlinear[:, 0], X_nonlinear[:, 1], c=y_nonlinear.ravel(), cmap='coolwarm', alpha=0.8)
plt.xlabel('Exam score (standardized)')
plt.ylabel('Interview score (standardized)')
plt.title('Admissions Dataset: Nonlinear Rule')
plt.show()


## Task 7. Review Activation and Loss Functions

Before building the MLP, you need a few standard functions.

In this task you will implement:
- `ReLU`
- `Softmax`
- `MSE`
- `Binary Cross-Entropy`
- `Hinge Loss`

Questions:
- Which of these are activation functions?
- Which of these are loss functions?
- Why is ReLU useful in hidden layers?


In [ ]:
def relu(x):
    # TODO 10: implement ReLU
    return ...


def relu_derivative(x):
    # TODO 11: implement the derivative of ReLU
    return ...


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def softmax(x):
    # TODO 12: shift values by the row-wise maximum for stability
    shifted = ...

    # TODO 13: exponentiate and normalize row-wise
    exp_x = ...
    return ...


def mse_loss(y_true, y_pred):
    # TODO 14: implement MSE
    return ...


def binary_cross_entropy(y_true, y_pred, eps=1e-8):
    # TODO 15: clip the predictions first
    y_pred = ...

    # TODO 16: implement BCE
    return ...


def hinge_loss(y_true_pm1, scores):
    # TODO 17: implement hinge loss
    return ...


## Task 8. Sanity-Check the Functions

Run the next cells after you complete the function definitions.

Question:
- Which loss becomes zero only when the classification margin is large enough?


In [ ]:
x_demo = np.linspace(-3, 3, 200).reshape(-1, 1)
relu_vals = relu(x_demo)
sigmoid_vals = sigmoid(x_demo)
softmax_demo = softmax(np.hstack([x_demo, -x_demo]))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(x_demo, relu_vals)
axes[0].set_title('ReLU')
axes[1].plot(x_demo, sigmoid_vals)
axes[1].set_title('Sigmoid')
axes[2].plot(x_demo, softmax_demo[:, 0], label='class 1')
axes[2].plot(x_demo, softmax_demo[:, 1], label='class 2')
axes[2].set_title('Softmax Output')
axes[2].legend()
plt.tight_layout()
plt.show()


In [ ]:
y_true_demo = np.array([[1.0], [0.0], [1.0], [0.0]])
y_pred_demo = np.array([[0.9], [0.2], [0.6], [0.4]])
scores_demo = np.array([[1.2], [-0.8], [0.3], [-0.1]])
y_pm1_demo = np.where(y_true_demo == 1, 1, -1)

print('MSE:', mse_loss(y_true_demo, y_pred_demo))
print('Binary Cross-Entropy:', binary_cross_entropy(y_true_demo, y_pred_demo))
print('Hinge Loss:', hinge_loss(y_pm1_demo, scores_demo))


## Task 9. Build an MLP for the Harder Admissions Rule

Now you will build a small MLP with:
- 2 input features
- 1 hidden layer with 8 neurons
- ReLU in the hidden layer
- 1 sigmoid output for binary classification

Questions:
- Why can this model represent more complex boundaries than the perceptron?
- What does the hidden layer add?


In [ ]:
def initialize_parameters(input_dim, hidden_dim, output_dim):
    params = {
        # TODO 18: initialize W1
        'W1': ...,
        # TODO 19: initialize b1
        'b1': ...,
        # TODO 20: initialize W2
        'W2': ...,
        # TODO 21: initialize b2
        'b2': ...,
    }
    return params


def forward_mlp(X, params):
    # TODO 22: first affine layer
    Z1 = ...

    # TODO 23: hidden activation
    A1 = ...

    # TODO 24: output affine layer
    Z2 = ...

    # TODO 25: output activation
    A2 = ...

    cache = {'X': X, 'Z1': Z1, 'A1': A1, 'Z2': Z2, 'A2': A2}
    return A2, cache


def backward_mlp(y, params, cache):
    m = y.shape[0]
    A2 = cache['A2']
    A1 = cache['A1']
    Z1 = cache['Z1']
    X = cache['X']

    # TODO 26: output layer gradients
    dZ2 = ...
    dW2 = ...
    db2 = ...

    # TODO 27: propagate gradients into the hidden layer
    dA1 = ...
    dZ1 = ...
    dW1 = ...
    db1 = ...

    grads = {'dW1': dW1, 'db1': db1, 'dW2': dW2, 'db2': db2}
    return grads


def update_parameters(params, grads, learning_rate):
    # TODO 28: update all four parameter tensors
    params['W1'] -= ...
    params['b1'] -= ...
    params['W2'] -= ...
    params['b2'] -= ...
    return params


## Task 10. Train the MLP

Now complete the main MLP training loop.

This loop should do the following in each epoch:
1. forward pass
2. compute binary cross-entropy loss
3. backward pass
4. update parameters
5. convert probabilities to class predictions

Question:
- How is this training loop similar to the ones used in modern deep learning frameworks?


In [ ]:
def train_mlp(X, y, hidden_dim=8, learning_rate=0.1, epochs=2000, print_every=200):
    params = initialize_parameters(X.shape[1], hidden_dim, 1)
    history = []

    for epoch in range(epochs):
        # TODO 29: forward pass
        y_pred, cache = ...

        # TODO 30: compute BCE loss
        loss = ...

        # TODO 31: backward pass
        grads = ...

        # TODO 32: update parameters
        params = ...

        # TODO 33: threshold probabilities at 0.5
        predictions = ...
        accuracy = np.mean(predictions == y)
        history.append((loss, accuracy))

        if epoch % print_every == 0:
            print(f'Epoch {epoch:4d} | loss={loss:.4f} | accuracy={accuracy:.4f}')

    return params, history


mlp_params, mlp_history = train_mlp(
    X_nonlinear,
    y_nonlinear,
    hidden_dim=8,
    learning_rate=0.1,
    epochs=2000,
    print_every=250,
)


## Task 11. Review the MLP Training Curves

Use the plots below to judge whether the model is learning.

Questions:
- Is the loss going down?
- Is the accuracy going up?
- Does the MLP seem more suitable for the nonlinear admissions task?


In [ ]:
losses = [l for l, _ in mlp_history]
accuracies = [a for _, a in mlp_history]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(losses, color='purple')
axes[0].set_title('MLP Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Cross-Entropy')

axes[1].plot(accuracies, color='darkgreen')
axes[1].set_title('MLP Training Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')

plt.tight_layout()
plt.show()


In [ ]:
final_probs, _ = forward_mlp(X_nonlinear, mlp_params)
final_preds = (final_probs >= 0.5).astype(int)
final_acc = np.mean(final_preds == y_nonlinear)
print('Final MLP accuracy:', final_acc)


## Task 12. Final Comparison

The final plot lets you compare the true nonlinear admissions labels with the MLP predictions.

Questions:
- Where does the model seem most accurate?
- Where does it still struggle?
- Why would the perceptron baseline have a harder time here?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X_nonlinear[:, 0], X_nonlinear[:, 1], c=y_nonlinear.ravel(), cmap='coolwarm', alpha=0.8)
axes[0].set_title('True Admissions Labels')
axes[1].scatter(X_nonlinear[:, 0], X_nonlinear[:, 1], c=final_preds.ravel(), cmap='coolwarm', alpha=0.8)
axes[1].set_title('MLP Predicted Labels')
plt.tight_layout()
plt.show()


## Final Reflection

By the end of this scenario, you should be able to explain:
- why perceptrons work for linearly separable problems
- why MLPs are stronger on nonlinear tasks
- how activation functions and loss functions fit into training
- how forward pass, backward pass, and parameter updates work together

Optional extension ideas:
- change the hidden layer size from 8 to 16
- replace ReLU with tanh
- add noise to the admissions labels
- compare BCE and hinge loss more carefully
- try a two-output softmax version of the classifier
